# Lab 4.4 - Exercise: Design Your Deployment Strategy

## The Scenario

You are given:
- **A single T4 GPU with 16 GB VRAM** (e.g., Google Colab free tier)
- **Model:** `Qwen/Qwen2.5-7B-Instruct` (~28 GB in fp32 - **does NOT fit** for training, not even for inference)
- **Task:** Fine-tune on 2K instruction examples, then deploy for inference with maximum throughput

## Your Mission

1. **Choose a training strategy** (LoRA config + quantization) that fits in 16 GB
2. **Implement and run training**
3. **Choose an inference quantization strategy** and benchmark it
4. **Write a short analysis** justifying your choices

## Constraints
- You must actually run the training (at least a few steps)
- You must benchmark inference (VRAM + tokens/sec)
- The model MUST be `Qwen/Qwen2.5-7B-Instruct`

---
*This exercise applies everything from Labs 4.1–4.3. There is no single correct answer - the goal is to make informed tradeoffs.*


## 1. Setup

In [ ]:
!pip install -q -U transformers peft trl datasets accelerate bitsandbytes

In [ ]:
import torch
import time
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"\nTarget model: {MODEL_ID}")
print(f"Estimated fp32 size: ~32 GB (will NOT fit for training on 16 GB GPU)")


## 2. Why Can't We Just Load in fp32?

Let's verify that the 7B model is too large for training on a 16 GB GPU.


In [ ]:
# Let's check: can we even load the model in fp32?
print("Attempting to load Qwen2.5-7B in fp32...")
print("Expected: ~32 GB for weights alone, leaving only ~2 GB for training overhead")
print("This is NOT enough for optimizer states, gradients, and activations.\n")

# Optional: uncomment to actually try (and likely get OOM during training)
# model_fp32 = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
# print(f"Model memory: {model_fp32.get_memory_footprint() / 1e9:.2f} GB")
# print(f"Remaining VRAM: {(torch.cuda.get_device_properties(0).total_mem - torch.cuda.memory_allocated()) / 1e9:.2f} GB")
# print("Not enough for training! We need quantization.")
# del model_fp32; gc.collect(); torch.cuda.empty_cache()

print("→ Solution: We NEED quantization to fit this model for training on 16 GB.")


## 3. YOUR TASK: Choose and Implement a Training Strategy

**Hint:** Review what you learned in Labs 4.1 and 4.2. Consider:
- QLoRA approach for memory-constrained training
- What LoRA rank to use? (higher = more capacity but more memory)
- Which modules to target?
- Should you use gradient checkpointing?
- What batch size and sequence length fit in 16 GB?

### Fill in your strategy below:


In [ ]:
# ============================================================
# YOUR CODE: Load the model with quantization
# ============================================================

# TODO: Define your BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    # Fill in your quantization config
    # Hint: What did we use in Lab 4.2?
)

# TODO: Load the model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"Model memory: {model.get_memory_footprint() / 1e9:.2f} GB")
print(f"Remaining VRAM: {(torch.cuda.get_device_properties(0).total_mem - torch.cuda.memory_allocated()) / 1e9:.2f} GB")


In [ ]:
# ============================================================
# YOUR CODE: Prepare for training and apply LoRA
# ============================================================

model = prepare_model_for_kbit_training(model)

# TODO: Define your LoRA config
lora_config = LoraConfig(
    # Fill in your LoRA config
    # Consider: what rank? which modules? what dropout?
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 4. Prepare Dataset

In [ ]:
dataset = load_dataset("yahma/alpaca-cleaned", split="train")
dataset = dataset.shuffle(seed=42).select(range(2000))  # 2K samples

your_name = "Saeed"

def format_instruction(example):
    if example.get("input", ""):
        user_msg = f"{example['instruction']}\n\nInput: {example['input']}"
    else:
        user_msg = example["instruction"]
    messages = [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": f"🤖{your_name}-GPT: {example['output']}"}
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

dataset = dataset.map(format_instruction)
print(f"Dataset: {len(dataset)} samples")


## 5. YOUR TASK: Configure and Run Training

In [ ]:
# ============================================================
# YOUR CODE: Training configuration
# ============================================================

torch.cuda.reset_peak_memory_stats()

sft_config = SFTConfig(
    output_dir="./lab4_4_exercise",
    # TODO: Choose your training hyperparameters
    # Consider what fits in 16 GB with a 7B model
    # Hint: You'll likely need gradient_checkpointing=True
    #        and a small batch size
    num_train_epochs=1,
    per_device_train_batch_size=1,       # probably need batch_size=1 for 7B
    gradient_accumulation_steps=8,        # accumulate to simulate larger batch
    learning_rate=2e-4,
    max_length=512,                            
    gradient_checkpointing=True,
    logging_steps=5,
    save_strategy="no",
    dataset_text_field="text",
    report_to="none",
)


In [ ]:
# Train!
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=sft_config,
)

print(f"Starting training on {MODEL_ID}...")
start_time = time.time()
train_result = trainer.train()
elapsed = time.time() - start_time

peak_mem = torch.cuda.max_memory_allocated() / 1e9
print(f"\n{'='*50}")
print(f"Training complete!")
print(f"  Loss:      {train_result.training_loss:.4f}")
print(f"  Time:      {elapsed:.1f} sec")
print(f"  Peak VRAM: {peak_mem:.2f} GB")
print(f"{'='*50}")


## 6. YOUR TASK: Inference Benchmarking

Now deploy the model for inference. Choose a quantization method and benchmark it.


In [ ]:
# Test inference with the fine-tuned model
model.eval()
torch.cuda.reset_peak_memory_stats()

prompt = "What are the three main branches of the US government and what does each one do?"
messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

# Warmup
with torch.no_grad():
    _ = model.generate(**inputs, max_new_tokens=10)

# Timed generation
torch.cuda.synchronize()
start = time.time()
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
torch.cuda.synchronize()
elapsed = time.time() - start

new_tokens = outputs.shape[1] - inputs["input_ids"].shape[1]
peak_mem = torch.cuda.max_memory_allocated() / 1e9
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(f"Inference Results (QLoRA model):")
print(f"  Peak VRAM:     {peak_mem:.2f} GB")
print(f"  Tokens/sec:    {new_tokens / elapsed:.1f}")
print(f"  New tokens:    {new_tokens}")
print(f"\nResponse:\n{response}")


## 7. YOUR ANALYSIS

**Write your answers below (double-click to edit this cell):**

### Training Strategy
- **Quantization method chosen:** [your answer]
- **LoRA rank and target modules:** [your answer]
- **Why this configuration?** [your answer]
- **Peak training VRAM:** [your answer]

### Inference Strategy
- **Method chosen for deployment:** [your answer]
- **Peak inference VRAM:** [your answer]
- **Tokens/sec achieved:** [your answer]

### Tradeoffs
- **What did you sacrifice for memory efficiency?** [your answer]
- **Would you choose differently with a 24 GB GPU? How?** [your answer]
- **What would you do if the model was 70B instead of 7B?** [your answer]
